# Khai thác tập phổ biến bằng Eclat (Julia): notebook demo

Notebook này tái lập toàn bộ kết quả của đồ án: cài đặt, kiểm thử, đối chiếu với SPMF, thực nghiệm Chương 2 và hai ứng dụng Chương 3.

**Cách chạy:** `Runtime → Run all`. Không cần đăng nhập ở bất kỳ bước nào.

- Colab chỉ hỗ trợ kernel Python, nên các ô gọi Julia qua lệnh shell (`!julia ...`), đúng hướng dẫn của đề.
- Mỗi phần chạy **độc lập**: phần sau tự tải những gì nó cần, nên lỗi ở một phần không làm hỏng phần khác.
- `CHE_DO = "nhanh"`: chạy lại một phần thực nghiệm rồi vẽ lại mọi hình từ CSV đã lưu trong repo. `CHE_DO = "day_du"`: chạy lại toàn bộ (lâu hơn nhiều).

In [ ]:
# === Cấu hình ===
REPO_URL = "https://github.com/<tai-khoan>/<Group_ID>.git"   # sửa thành repo của nhóm
REPO_REF = "main"        # khi nộp bài: đổi thành tag "v1.0"
CHE_DO = "nhanh"         # "nhanh" hoặc "day_du"

## 1. Cài đặt môi trường

In [ ]:
import os
if not os.path.exists("/content/repo"):
    !git clone --depth 1 --branch {REPO_REF} {REPO_URL} /content/repo
%cd /content/repo
!git log -1 --format="Commit: %h  %s"

Cài Julia 1.11.4 đúng theo hướng dẫn của đề:

In [ ]:
!curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz -o julia.tar.gz
!tar -xzf julia.tar.gz -C /usr --strip-components 1
!julia --version

Cài Java 25 (bản `spmf.jar` hiện tại cần Java ≥ 25) để chạy SPMF, cài đặt tham chiếu dùng khi đối chiếu kết quả:

In [ ]:
!curl -fsSL "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jre/hotspot/normal/eclipse?project=jdk" -o jre.tar.gz
!mkdir -p /opt/jre25 && tar -xzf jre.tar.gz -C /opt/jre25 --strip-components 1
os.environ["JAVA"] = "/opt/jre25/bin/java"
!$JAVA -version

In [ ]:
%%time
!julia --project=. -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()'
!julia --project=. scripts/fetch_spmf.jl
!julia --project=. scripts/env_info.jl && cat results/ENV.md

## 2. Dữ liệu

Tải 5 bộ benchmark và kiểm tra SHA-256 (nguồn và thống kê: `data/benchmark/README.md`).

In [ ]:
!julia --project=. scripts/fetch_data.jl benchmark
!julia --project=. scripts/data_stats.jl      # thống kê đo lại, so với bảng của đề

## 3. Demo giao diện dòng lệnh

Chạy **đúng lệnh mẫu của đề** (`demo.jl --minsup 0.01 data/benchmark.txt`), với `data/benchmark.txt` là bộ Retail.
Kết quả ghi theo định dạng SPMF vào `output.txt`.

In [ ]:
!julia --project=. demo.jl --minsup 40% data/toy/pasquier99.txt -o -      # dữ liệu toy: đúng 15 tập
!cp data/benchmark/retail.txt data/benchmark.txt
!julia --project=. demo.jl --minsup 0.01 data/benchmark.txt
!head -5 output.txt

Đối chiếu với SPMF (cùng chuỗi minsup cho cả hai bên):

In [ ]:
!julia --project=. scripts/compare_spmf.jl data/benchmark/chess.txt 80%
!julia --project=. scripts/compare_spmf.jl data/benchmark/mushroom.txt 20%

## 4. Kiểm thử tự động

Gồm: đáp án SPMF cho toy và ca biên, oracle duyệt trâu trên 200 CSDL ngẫu nhiên, kiểm tra hiệu năng, và đối chiếu dấu vân tay trên 5 bộ benchmark.

In [ ]:
%%time
!julia --project test/runtests.jl

## 5. Chương 2: Thực nghiệm và đánh giá

Số liệu chính thức trong báo cáo được đo trên một máy cố định (cấu hình ở `results/ENV.md`), theo giao thức ở mục 8.5 của kế hoạch: mỗi lần đo một tiến trình mới, có chạy khởi động, lặp 3 lần, lấy trung vị. Kết quả thô nằm trong `results/*.csv`.

- **Chế độ nhanh:** chạy lại một phần E1, E2, E4, E5 và E8 (Chess, Retail; mỗi phép đo 1 lần) vào thư mục riêng `results_colab/`, để không ghi đè số liệu chính thức. Thời gian trên Colab sẽ khác máy đo chính thức, nhưng kết quả khai thác phải khớp SPMF 100%.
- **Chế độ đầy đủ:** chạy lại toàn bộ E1–E10 (mất vài giờ).

In [ ]:
%%time
if CHE_DO == "nhanh":
    os.environ["FIM_RESULTS_DIR"] = "/content/repo/results_colab"
    !julia --project=. scripts/subsample.jl retail
    !julia --project=. scripts/bench.jl E1 E2 E4 E5 --quick --datasets chess,retail
    !julia --project=. scripts/bench_ablation.jl --repeats 1 --datasets chess,retail
else:
    os.environ.pop("FIM_RESULTS_DIR", None)
    !julia --project=. scripts/fetch_data.jl benchmark
    !julia --project=. scripts/subsample.jl retail
    !julia --project=. scripts/subsample.jl accidents
    !julia --project=. scripts/gen_synthetic.jl
    !julia --project=. scripts/bench.jl all
    !julia --project=. scripts/bench_cli.jl
    !julia --project=. scripts/bench_ablation.jl
    !julia --project=. scripts/bench_density.jl
    !julia --project=. scripts/bench_memory_trace.jl
os.environ.pop("FIM_RESULTS_DIR", None)

Tính đúng (E1) và ablation (E8) trên số liệu chính thức:

In [ ]:
import pandas as pd
from IPython.display import Markdown, Image, display
e1 = pd.read_csv("results/e1_correctness.csv")
print(f"E1: {e1['fingerprint_match'].sum()}/{len(e1)} cặp (bộ dữ liệu, minsup, phiên bản) khớp SPMF 100%")
display(e1.groupby("dataset")["fingerprint_match"].agg(["sum", "count"]))
display(Markdown(open("results/ablation_B03.md", encoding="utf-8").read()))
if CHE_DO == "nhanh" and os.path.exists("results_colab/e1_correctness.csv"):
    q = pd.read_csv("results_colab/e1_correctness.csv")
    print(f"Chạy lại trên Colab: {q['fingerprint_match'].sum()}/{len(q)} khớp SPMF")

Vẽ lại mọi hình của Chương 2 từ CSV chính thức:

In [ ]:
%%time
!julia --project=scripts -e 'using Pkg; Pkg.instantiate()'
!julia --project=scripts scripts/plots.jl
!julia --project=scripts scripts/plots_a.jl
for f in ["F03_thoi_gian_theo_minsup", "F04_so_tap_theo_minsup", "F05_bo_nho", "F06_kha_nang_mo_rong",
          "F07_do_dai_giao_dich", "F08_mat_do_bitset_vector", "F09_so_voi_apriori_fpgrowth",
          "F10_bo_nho_theo_tien_trinh"]:
    print(f); display(Image(f"results/figures/{f}.png", width=900))

## 6. Ứng dụng 1: Phân tích giỏ hàng (Groceries)

Mỗi hóa đơn là một giao dịch, mỗi mặt hàng là một item (tên lưu trong dòng `@ITEM=` theo định dạng SPMF). Luật kết hợp sinh từ tập phổ biến do Eclat của nhóm tìm ra; xếp theo lift; kiểm tra độ ổn định trên 20% dữ liệu giữ lại.

In [ ]:
%%time
!julia --project=. scripts/fetch_data.jl groceries
!julia --project=. apps/groceries/prepare.jl > /dev/null
!julia --project=. apps/groceries/analyze.jl > /dev/null
for f in ["B08_bien_doi", "B08b_nguong", "B09_top10_lift", "B09b_top10_conf", "B10_on_dinh", "B08c_phuong_an"]:
    display(Markdown(open(f"results/apps/groceries/{f}.md", encoding="utf-8").read()))
!julia --project=scripts scripts/plots.jl > /dev/null
display(Image("results/figures/F11_luat_groceries.png", width=600))

## 7. Ứng dụng 2: Mẫu trực quan trên ảnh (bag of visual words)

4 lớp của Caltech-101, mỗi lớp 100 ảnh (70 huấn luyện, 30 kiểm tra). Descriptor gradient và k-means++ do nhóm tự viết; mỗi ảnh thành một giao dịch các visual word; khai thác bằng Eclat của nhóm; tìm mẫu đặc trưng từng lớp (tỉ lệ tăng) và dùng chúng làm đặc trưng cho bộ phân lớp.

In [ ]:
%%time
!julia --project=apps -e 'using Pkg; Pkg.instantiate()'
!julia --project=. scripts/fetch_data.jl caltech
!julia --project=apps apps/bovw/prepare.jl
!julia --project=apps apps/bovw/analyze.jl > /dev/null
!julia --project=apps apps/bovw/figures.jl
for f in ["B11_bien_doi", "B12_mau_dac_trung", "B13_phan_lop"]:
    display(Markdown(open(f"results/apps/bovw/{f}.md", encoding="utf-8").read()))
display(Image("results/figures/F12_mau_truc_quan_tren_anh.png", width=1000))
display(Image("results/figures/F13_chan_dung_visual_word.png", width=900))

## 8. Tổng kết

- Báo cáo đầy đủ: `docs/Report.pdf`.
- Số liệu thô: `results/*.csv`; bảng: `results/*.md`, `results/apps/`; hình: `results/figures/`.
- Kế hoạch và quy trình làm việc của nhóm: lịch sử commit của repo.